In [31]:
from geopy.geocoders import Nominatim
import pandas as pd
import geopandas as gpd
import h3pandas
import time
from shapely.geometry import Point

In [32]:
geolocator = Nominatim(user_agent="rais-geocoding")

In [33]:
YEAR = "2021"

jobs = pd.read_csv(f"../data/address/reallyNotFound__{YEAR}.csv", sep=';')
jobs = jobs.groupby(["nome_logradouro", "NUM_ENDERECO", "bairro"]).agg({"estoque": "sum", "_index": "first"}).reset_index()

In [34]:
jobs.head(2)

,nome_logradouro,NUM_ENDERECO,bairro,estoque,_index
0,R,400.0,BUTANTA,1,173009
1,ITU,483.0,CERQUEIRA CESAR,1,24344


In [ ]:
_count = 0
notFound = 0
found = 0

locations = []

for i, row in jobs.iterrows():
    address = str(row.nome_logradouro) + ", " + str(row.NUM_ENDERECO) + ", " + str(row.bairro) + ", " + "São Paulo, São Paulo, Brasil"
    try:
        location = geolocator.geocode(address)
        locations.append({"latitude": location.latitude, "longitude": location.longitude, "_index": row["_index"]})
        found += 1
    except:
        notFound += 1
        if notFound%30 == 0: 
            print("****")
            print(_count, 'not found... while found', found)
            print()

    _count += 1
    if _count%120 == 0: print(_count, 'has gone...', found, 'worked')
    #if _count==50: break
    #print(_count, 'keep waiting...')
    #break

    time.sleep(0.8)


****
70 not found... while found 41

120 has gone... 68 worked
****
136 not found... while found 77

240 has gone... 152 worked
****
269 not found... while found 180

360 has gone... 244 worked
****
363 not found... while found 244

****
450 not found... while found 301

480 has gone... 322 worked
****
532 not found... while found 353

****
592 not found... while found 383

600 has gone... 383 worked
****
648 not found... while found 409

720 has gone... 451 worked
****
724 not found... while found 455

****
811 not found... while found 512

840 has gone... 520 worked
****
856 not found... while found 527

****
916 not found... while found 557

960 has gone... 581 worked
****
997 not found... while found 608

****
1054 not found... while found 635

1080 has gone... 649 worked
****
1108 not found... while found 659

****
1159 not found... while found 680

1200 has gone... 698 worked
****
1223 not found... while found 714

****
1296 not found... while found 757

1320 has gone... 764 work

In [36]:
newLocDf = pd.DataFrame(locations)

In [37]:
foundJobs = jobs.merge(newLocDf, on="_index")
len(foundJobs), foundJobs.estoque.sum()

(2805, 549704)

In [38]:
len(foundJobs), len(jobs), foundJobs.estoque.sum()

(2805, 7034, 549704)

In [39]:
reallyNotFound2 = jobs[~jobs["_index"].isin(foundJobs["_index"])]
reallyNotFound2.to_csv(f"../data/address/reallyNotFound2__{YEAR}.csv", index=False, sep=';')

In [40]:
reallyNotFound2.estoque.sum(), jobs.estoque.sum()

(335833, 885537)

In [41]:
foundJobs['geometry'] = foundJobs.apply(lambda _row: Point(_row.longitude, _row.latitude), axis=1)
foundJobs = gpd.GeoDataFrame(foundJobs)
foundJobs = foundJobs.h3.geo_to_h3(9).reset_index()

foundJobs = foundJobs.rename(columns={"estoque": "T001", "h3_09": 'id_hex'})
print(foundJobs.T001.sum(), jobs.estoque.sum())
print('now building csv:::: almost done!')
#merged.to_file(f"./data/address/origins__{YEAR}.geojson", driver='GeoJSON')
foundJobs.drop(columns=['geometry']).to_csv(f"../data/address/_destinations2__{YEAR}.csv", sep=';', index=False)

549704 885537
now building csv:::: almost done!
